# AC-MOT v10 — SOTA Comparison
**OC-SORT + BoT-SORT vs Baseline vs AC-MOT**

- Detector fixed: YOLOv8n (same as all other runs)
- conf=0.25, imgsz=640, FP16 (same as Baseline_Default)
- Sequences: same 12 valid VisDrone sequences
- Est. time: ~35 min on T4 (2 trackers × 12 seqs)
- Saves to: `/content/drive/MyDrive/VisDrone_Results/`

In [ ]:
!pip install ultralytics motmetrics opencv-python-headless pandas numpy tqdm lap pyyaml -q

import time, shutil, gc
from pathlib import Path
from datetime import datetime

import cv2, numpy as np, pandas as pd, torch, motmetrics as mm
from tqdm import tqdm
from ultralytics import YOLO
from google.colab import drive

try: torch.backends.cudnn.benchmark = True
except: pass

drive.mount('/content/drive', force_remount=False)

DATASET_ROOT  = Path('/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev')
SEQ_DIR       = DATASET_ROOT / 'sequences'
ANNOT_DIR     = DATASET_ROOT / 'annotations'
DRIVE_RESULTS = Path('/content/drive/MyDrive/VisDrone_Results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
LOCAL_TMP     = Path('/content/_sota_tmp')

assert SEQ_DIR.exists(), f'Dataset not found: {SEQ_DIR}'
all_sequences = sorted([d for d in SEQ_DIR.iterdir() if d.is_dir()])

MODEL_NAME = 'yolov8n.pt'
DEVICE     = '0' if torch.cuda.is_available() else 'cpu'
HALF       = DEVICE != 'cpu'
CONF       = 0.25   # same as Baseline_Default
IOU        = 0.45
IMGSZ      = 640

# Systems to run
SOTA_SYSTEMS = [
    dict(name='OC-SORT',  tracker='ocsort.yaml'),
    dict(name='BoT-SORT', tracker='botsort.yaml'),
]

print(f'Device={DEVICE} | FP16={HALF} | conf={CONF} | imgsz={IMGSZ}')
print(f'Sequences available: {len(all_sequences)}')
print('Systems:', [s["name"] for s in SOTA_SYSTEMS])

In [ ]:
# ── Helper functions ─────────────────────────────────────────────

def load_gt(path):
    if not path.exists(): return pd.DataFrame()
    cols = ['frame','id','x','y','w','h','score','cat','trunc','occ']
    df = pd.read_csv(path, header=None, names=cols)
    df = df[df['cat'].isin([1,4,5,6,9])]
    df = df[(df['occ']<2)&(df['trunc']<2)&(df['score']==1)]
    return df.reset_index(drop=True)

def iou_dist(pred, gt):
    if not len(pred) or not len(gt): return np.empty((len(gt),len(pred)))
    ix1=np.maximum(pred[:,0:1].T,gt[:,0:1]); iy1=np.maximum(pred[:,1:2].T,gt[:,1:2])
    ix2=np.minimum(pred[:,2:3].T,gt[:,2:3]); iy2=np.minimum(pred[:,3:4].T,gt[:,3:4])
    inter=np.maximum(0,ix2-ix1)*np.maximum(0,iy2-iy1)
    ap=(pred[:,2]-pred[:,0])*(pred[:,3]-pred[:,1])
    ag=(gt[:,2]-gt[:,0])*(gt[:,3]-gt[:,1])
    union=ap[np.newaxis,:]+ag[:,np.newaxis]-inter
    return 1.0-np.where(union>0,inter/union,0.0)

def hota_approx(tp,fp,fn,ids):
    return float(np.sqrt(tp/max(tp+fp+fn,1)*max(0.0,1.0-ids/max(tp,1))))

def eval_acc(acc, name='seq'):
    mh=mm.metrics.create()
    s=mh.compute(acc,metrics=['mota','idf1','num_switches','recall','precision',
                               'num_misses','num_false_positives','num_matches'],name=name)
    r=s.iloc[0]
    return dict(mota=float(r['mota']),idf1=float(r['idf1']),recall=float(r['recall']),
                precision=float(r['precision']),ids=int(r['num_switches']),
                fn=int(r['num_misses']),fp=int(r['num_false_positives']),
                matches=int(r['num_matches']),
                hota=hota_approx(int(r['num_matches']),int(r['num_false_positives']),
                                 int(r['num_misses']),int(r['num_switches'])))

def reset_tracker(model):
    if getattr(model,'predictor',None) is not None: model.predictor = None

print('Helpers ready')

In [ ]:
# CELL 3 — fixed: BoT-SORT only
SOTA_SYSTEMS = [
    dict(name='BoT-SORT', tracker='botsort.yaml'),
]

ts      = datetime.now().strftime('%Y%m%d_%H%M%S')
run_tag = f'acmot_v10_SOTA_{ts}'
all_rows = []

for system in SOTA_SYSTEMS:
    model = YOLO(MODEL_NAME)
    if HALF: model.model.half()
    rows = []

    for seq in tqdm(all_sequences, desc=system['name']):
        gt = load_gt(ANNOT_DIR / f'{seq.name}.txt')
        if gt.empty or not list(seq.glob('*.jpg')): continue

        LOCAL_TMP.mkdir(exist_ok=True)
        local_seq = LOCAL_TMP / seq.name
        if local_seq.exists(): shutil.rmtree(local_seq)
        shutil.copytree(seq, local_seq)
        frames = sorted(local_seq.glob('*.jpg'))

        reset_tracker(model)
        acc   = mm.MOTAccumulator(auto_id=True)
        times = []

        for idx, fp in enumerate(frames, start=1):
            t0  = time.perf_counter()
            img = cv2.imread(str(fp))
            if img is None: continue

            res = model.track(source=img, tracker=system['tracker'],
                              conf=CONF, iou=IOU, imgsz=IMGSZ,
                              half=HALF, persist=True, verbose=False, device=DEVICE)
            times.append(time.perf_counter()-t0)

            pred_ids   = res[0].boxes.id.cpu().numpy().astype(int) if res[0].boxes.id is not None else np.array([],dtype=int)
            pred_boxes = res[0].boxes.xyxy.cpu().numpy()           if res[0].boxes.id is not None else np.empty((0,4))

            gt_f     = gt[gt['frame']==idx]
            gt_ids   = gt_f['id'].values
            gt_boxes = (np.column_stack([gt_f['x'].values,gt_f['y'].values,
                                         gt_f['x'].values+gt_f['w'].values,
                                         gt_f['y'].values+gt_f['h'].values])
                        if len(gt_f) else np.empty((0,4)))
            dist = iou_dist(pred_boxes, gt_boxes)
            acc.update(gt_ids, pred_ids, dist if dist.size else np.empty((len(gt_ids),len(pred_ids))))

        shutil.rmtree(local_seq, ignore_errors=True)
        m   = eval_acc(acc, seq.name)
        fps = 1.0/np.mean(times) if times else 0.0
        rows.append(dict(run_tag=run_tag, system=system['name'], sequence=seq.name,
                         frames=len(frames), fps=round(fps,2), **m))
        tqdm.write(f"BoT-SORT {seq.name[:28]:28s} MOTA={m['mota']:.3f} IDF1={m['idf1']:.3f} IDS={m['ids']:4d} FPS={fps:.1f}")

    if HALF: torch.cuda.empty_cache()
    gc.collect()
    all_rows.extend(rows)
    df_sys = pd.DataFrame(rows)
    print(f"\nBoT-SORT MEAN: MOTA={df_sys['mota'].mean():.4f}  IDF1={df_sys['idf1'].mean():.4f}  IDS={df_sys['ids'].sum()}  FPS={df_sys['fps'].mean():.1f}\n")

df_all   = pd.DataFrame(all_rows)
out_path = DRIVE_RESULTS / f'{run_tag}_per_sequence.csv'
df_all.to_csv(out_path, index=False)
print(f'Saved -> {out_path}')

In [ ]:
# ── FINAL COMPARISON TABLE ────────────────────────────────────────
# Paste Baseline_Default results here for comparison
baseline_data = {
    'system': 'Baseline_Default',
    'mota': 0.1707, 'idf1': 0.2874, 'ids': 463, 'fps': 22.6
}
acmot_data = {
    'system': 'AC-MOT_v10 (ours)',
    'mota': 0.1955, 'idf1': 0.3244, 'ids': 328, 'fps': 21.1
}

summary_rows = [baseline_data, acmot_data]
for sys_name in df_all['system'].unique():
    g = df_all[df_all['system']==sys_name]
    summary_rows.append(dict(
        system=sys_name,
        mota=round(g['mota'].mean(),4),
        idf1=round(g['idf1'].mean(),4),
        ids=int(g['ids'].sum()),
        fps=round(g['fps'].mean(),1)
    ))

df_summary = pd.DataFrame(summary_rows)
df_summary['mota_delta'] = df_summary['mota'] - float(df_summary.iloc[0]['mota'])
df_summary['ids_delta']  = df_summary['ids']  - int(df_summary.iloc[0]['ids'])

print('\n' + '='*90)
print('FULL COMPARISON: YOLOv8n backbone, VisDrone2019-MOT-test-dev, 12 sequences')
print('='*90)
print(df_summary[['system','mota','idf1','ids','fps','mota_delta','ids_delta']]
      .to_string(index=False, float_format=lambda x: f'{x:.4f}'))
print('='*90)

# Save summary
sum_path = DRIVE_RESULTS / f'{run_tag}_full_comparison.csv'
df_summary.to_csv(sum_path, index=False)
print(f'Saved -> {sum_path}')